# 01 · Running the experiment

**Your job in one line: produce the numbers the paper reports.**

You do not need to understand the agent's internals to do this well. You need to
run it correctly, repeatedly, and record what you ran.

If you want the background first, read `../rashan-table-agent/01_the_data.ipynb`
and `04_the_agent.ipynb`.

> **Note.** The provider-check cell below needs a key in the cookbook-root
> `.env`. It is left unexecuted here because both providers were unavailable
> when this was committed. Everything else in this notebook is instructions.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `run_eval.py` | runs configurations and scores them against gold | `--configs baseline,agent` |
| `--repeats` | how many times to run everything | `--repeats 3` |
| `--sample-seed` | take a representative random sample | `--sample-seed 7` |
| `BUDGET_USD` | hard spend cap; stops and saves partial results | `BUDGET_USD=5` |

## Step 1 — check your key is loaded and see the ceiling

Never start a long run without confirming which model you are about to pay for.

In [1]:
import os, sys
sys.path.insert(0, "../rashan-table-agent")
import agent as base
base._load_env_file()
p = base.resolve_provider()
print("provider :", p)
print("model    :", base.OPENAI_MODEL if p == "openai" else base.GROQ_MODEL)
print("ceiling  : $", os.environ.get("BUDGET_USD", "2.0 (default)"))

provider : groq
model    : openai/gpt-oss-20b
ceiling  : $ 2.0


## Step 2 — the three commands you will run

Run these in a terminal from `02-experiments/rashan-table-agent/`, not here —
they take hours and a notebook is a bad place for that.

```bash
# 1. main matrix, repeat 1 of 3
BUDGET_USD=5 python run_eval.py --configs baseline,cot,agent \
    --limit 460 --repeats 1 --sample-seed 7
#    ... then repeats 2 and 3 as separate jobs, copying results/ aside each time

# 2. transfer arm, a second subject area
BUDGET_USD=5 python run_eval.py --domain CompSci --configs baseline,agent \
    --limit 400 --repeats 3 --sample-seed 7
```

**Before each re-run, copy `results/*.csv` somewhere safe.** The harness
overwrites them, and an overwritten result is a lost result.

## Step 3 — what to record for every run

Put this in `results/RUN_LOG.md`. A number nobody can trace is not usable in a
paper.

```
 date | command exactly as typed | model | N questions | repeats
 | total spend printed at the end | anything that went wrong
```

## Step 4 — if a run stops early

That is the budget ceiling doing its job. It saves whatever finished and prints
where it stopped. Record the number it stopped at and tell Rashan — do not just
raise the ceiling and rerun blindly.

## What "done" looks like

```
 · results/*.csv committed for every configuration and repeat
 · results/RUN_LOG.md filled in
 · the spend reported for each run
```

Next: `02_analyse_results.ipynb`.